SETUP

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

df_inner = ('../bases/tabelas_unificadas/Base_Unificada_Tratada.csv')
df_outer = ('../bases/tabelas_unificadas/Base_Unificada_Outer.csv')

# Paleta consistente para todo o notebook
PALETTE = {0: "#4C72B0", 1: "#C44E52"}  # 0 = não-churn, 1 = churn
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

VISAO GERAL E COMPARATIVA 

In [ ]:
def overview(df, nome="dataset"):
    print(f"===== {nome.upper()} =====")
    print(f"Shape: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
    print(f"Memória: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"\nTipos de dados:\n{df.dtypes.value_counts()}")
    
    resumo = pd.DataFrame({
        "dtype": df.dtypes,
        "n_unicos": df.nunique(),
        "missing_n": df.isnull().sum(),
        "missing_%": (df.isnull().sum() / len(df) * 100).round(2)
    }).sort_values("missing_%", ascending=False)
    
    return resumo

resumo_inner = overview(df_inner, "inner merge")
resumo_outer = overview(df_outer, "outer merge")
resumo_outer  # ver colunas com mais missing

DECISAO ENTRE INNER OU OUTER

In [ ]:
diff_linhas = len(df_outer) - len(df_inner)
pct_diff = diff_linhas / len(df_outer) * 100

print(f"Outer tem {diff_linhas:,} linhas extras em relação à inner ({pct_diff:.1f}% da base outer)")

# Quem são essas linhas extras? (assumindo uma coluna chave, ex: 'cliente_id')
ids_so_outer = set(df_outer["cliente_id"]) - set(df_inner["cliente_id"])
extras = df_outer[df_outer["cliente_id"].isin(ids_so_outer)]

print(f"\nMissing médio nessas linhas extras vs base completa:")
print(f"Linhas extras: {extras.isnull().mean().mean()*100:.1f}% missing médio")
print(f"Base outer completa: {df_outer.isnull().mean().mean()*100:.1f}% missing médio")

# Essas linhas extras têm informação de churn (target)?
print(f"\n% com target preenchido nas linhas extras: {extras['churn'].notna().mean()*100:.1f}%")

ANALISES DOS VALORES NULOS

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
missing_pct = resumo_outer["missing_%"].sort_values(ascending=True)
missing_pct = missing_pct[missing_pct > 0]

ax.barh(missing_pct.index, missing_pct.values, color="#C44E52")
ax.set_xlabel("% Missing")
ax.set_title("Percentual de valores faltantes por coluna (base outer)")
plt.tight_layout()
plt.show()

# matriz visual rápida (amostra para performance)
msno.matrix(df_outer.sample(min(5000, len(df_outer)), random_state=42))
plt.show()

DUPLICATAS E INCONSISTÊNCIAS

In [ ]:
print(f"Linhas duplicadas (todas colunas): {df_outer.duplicated().sum()}")
print(f"IDs de cliente duplicados: {df_outer['cliente_id'].duplicated().sum()}")

# checagens de sanidade — ajuste para as colunas reais da sua base
checagens = {
    "idade_negativa": (df_outer.get("idade", pd.Series(dtype=float)) < 0).sum(),
    "idade_implausivel": (df_outer.get("idade", pd.Series(dtype=float)) > 100).sum(),
    "premio_zero_ou_negativo": (df_outer.get("premio", pd.Series(dtype=float)) <= 0).sum(),
}
pd.Series(checagens)